In [1]:
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

from evaluation_utils import evaluate_model_dataset

ARTIFACT_DIR = Path("artifacts")
SEQUENCES_PATH = ARTIFACT_DIR / "sequences_and_splits.pkl"
CANDIDATES_PATH = ARTIFACT_DIR / "shared_evaluation_candidates.pkl"

with open(SEQUENCES_PATH, "rb") as f:
    data = pickle.load(f)

with open(CANDIDATES_PATH, "rb") as f:
    shared_candidates = pickle.load(f)

train_sequences = data["train_sequences"]
test_targets = data["test_targets"]
test_eval_users = data["test_eval_users"]

test_candidate_sets = shared_candidates["test_candidate_sets"]
EVAL_K = shared_candidates["eval_k"]

assert set(test_candidate_sets) == set(test_eval_users)
assert all(len(s) == shared_candidates["n_negatives"] + 1 for s in test_candidate_sets.values())
assert all(s.count(test_targets[u]) == 1 for u, s in test_candidate_sets.items())

print("Shared artifacts loaded.")
print(f"Train users: {len(train_sequences):,}")
print(f"Test users: {len(test_eval_users):,}")
print(f"Eval k: {EVAL_K}")

Shared artifacts loaded.
Train users: 246,307
Test users: 35,637
Eval k: 10


In [3]:
## 1. Recompute popularity scores (train-only, identical to Step2)

'''This must match `Step2_CandidateSampling_PopularityBaseline.ipynb` exactly: popularity
is purchase frequency within `train_sequences` only. We recompute it here rather than
reload it, since it is cheap and this keeps this notebook fully self-contained.'''

'This must match `Step2_CandidateSampling_PopularityBaseline.ipynb` exactly: popularity\nis purchase frequency within `train_sequences` only. We recompute it here rather than\nreload it, since it is cheap and this keeps this notebook fully self-contained.'

In [4]:
item_popularity = Counter(
    item
    for sequence in train_sequences.values()
    for item in sequence
)

def score_candidates_popularity(candidates, item_popularity):
    """Assign training purchase counts to candidate products."""
    return [
        item_popularity.get(item, 0)
        for item in candidates
    ]

print(f"Products with training purchases: {len(item_popularity):,}")

Products with training purchases: 356,014


In [5]:
## 2. Score and evaluate the test candidates

'''Same scoring function and same `evaluate_model_dataset` call as the validation
evaluation in Step2, just pointed at `test_candidate_sets` / `test_targets`.'''

'Same scoring function and same `evaluate_model_dataset` call as the validation\nevaluation in Step2, just pointed at `test_candidate_sets` / `test_targets`.'

In [6]:
test_model_scores = {
    user: score_candidates_popularity(candidates, item_popularity)
    for user, candidates in test_candidate_sets.items()
}

test_summary, test_results = evaluate_model_dataset(
    candidate_sets=test_candidate_sets,
    model_scores=test_model_scores,
    targets=test_targets,
    k=EVAL_K
)

print("Popularity Baseline — Test Results")
print("-" * 40)
display(pd.DataFrame([test_summary]))

Popularity Baseline — Test Results
----------------------------------------


,n_users,HitRate@10,NDCG@10,MRR
0,35637,0.303673,0.198241,0.178813


In [7]:
## 3. Performance by training item coverage (test set)

'''
Same seen/unseen breakdown as the validation notebook's Section 5.2, run on the test
split. Expect the unseen share to be higher here than on validation, since the test
window sits later and further from the training cutoff -- worth comparing directly.'''

"\nSame seen/unseen breakdown as the validation notebook's Section 5.2, run on the test\nsplit. Expect the unseen share to be higher here than on validation, since the test\nwindow sits later and further from the training cutoff -- worth comparing directly."

In [8]:
train_items = {
    item
    for sequence in train_sequences.values()
    for item in sequence
}

test_results["target_seen_in_train"] = (
    test_results["target_item"].isin(train_items)
)

coverage_results = (
    test_results
    .groupby("target_seen_in_train")
    .agg(
        n_users=("user_id", "count"),
        hit_rate_10=(f"HitRate@{EVAL_K}", "mean"),
        ndcg_10=(f"NDCG@{EVAL_K}", "mean"),
        mrr=("RR", "mean")
    )
    .reset_index()
)

coverage_results["target_group"] = coverage_results["target_seen_in_train"].map({
    True: "Seen in training",
    False: "Unseen in training"
})

display(
    coverage_results[["target_group", "n_users", "hit_rate_10", "ndcg_10", "mrr"]]
)

n_unseen_test = int((~test_results["target_seen_in_train"]).sum())
print(f"\nUnseen-in-training share (test): {n_unseen_test / len(test_results) * 100:.2f}%")

,target_group,n_users,hit_rate_10,ndcg_10,mrr
0,Unseen in training,17710,0.00000,0.000000,0.011101
1,Seen in training,17927,0.60367,0.394082,0.344495



Unseen-in-training share (test): 49.70%


In [9]:
## 4. Save test results

'''Saved under the same naming convention as the validation files, so they slot directly
into the existing `artifacts/` directory and the final comparison table.'''

'Saved under the same naming convention as the validation files, so they slot directly\ninto the existing `artifacts/` directory and the final comparison table.'

In [10]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

summary_path = ARTIFACT_DIR / "popularity_test_summary.csv"
pd.DataFrame([test_summary]).to_csv(summary_path, index=False)

results_path = ARTIFACT_DIR / "popularity_test_user_results.csv"
test_results.to_csv(results_path, index=False)

print("Saved files:")
for path in [summary_path, results_path]:
    print(f"- {path} ({path.stat().st_size / 1e6:.2f} MB)")

Saved files:
- artifacts\popularity_test_summary.csv (0.00 MB)
- artifacts\popularity_test_user_results.csv (1.67 MB)


In [11]:
## 5. Verify saved artifacts

'''Same reload-and-compare pattern as Step2's Section 6.2, so a broken write fails loudly
here rather than silently downstream in the final comparison table.'''

"Same reload-and-compare pattern as Step2's Section 6.2, so a broken write fails loudly\nhere rather than silently downstream in the final comparison table."

In [12]:
saved_summary = pd.read_csv(summary_path)
saved_user_results = pd.read_csv(results_path)

for metric in [f"HitRate@{EVAL_K}", f"NDCG@{EVAL_K}", "MRR"]:
    assert np.isclose(saved_summary.loc[0, metric], test_summary[metric])

assert len(saved_user_results) == len(test_eval_users)

print("Saved test summary verified.")
print("Saved per-user test results verified.")
print("\nAll artifact checks passed.")

Saved test summary verified.
Saved per-user test results verified.

All artifact checks passed.


In [13]:
## 6. Updated comparison table (val + test, all models)

'''Pulls together popularity (val + test, now complete), plus whatever other models'
summaries already exist in `artifacts/`, into one table for the final report.'''

"Pulls together popularity (val + test, now complete), plus whatever other models'\nsummaries already exist in `artifacts/`, into one table for the final report."

In [14]:
rows = []

pop_val_path = ARTIFACT_DIR / "popularity_val_summary.csv"
if pop_val_path.exists():
    pop_val = pd.read_csv(pop_val_path).iloc[0]
    rows.append({"model": "Popularity", "split": "val",
                 "HitRate@10": pop_val["HitRate@10"], "NDCG@10": pop_val["NDCG@10"], "MRR": pop_val["MRR"]})

rows.append({"model": "Popularity", "split": "test",
             "HitRate@10": test_summary[f"HitRate@{EVAL_K}"],
             "NDCG@10": test_summary[f"NDCG@{EVAL_K}"],
             "MRR": test_summary["MRR"]})

lightfm_val_path = ARTIFACT_DIR / "lightfm_val_summary.csv"
lightfm_test_path = ARTIFACT_DIR / "lightfm_test_summary.csv"
for path, split in [(lightfm_val_path, "val"), (lightfm_test_path, "test")]:
    if path.exists():
        r = pd.read_csv(path).iloc[0]
        rows.append({"model": "LightFM (purchase-only)", "split": split,
                     "HitRate@10": r["HitRate@10"], "NDCG@10": r["NDCG@10"], "MRR": r["MRR"]})

sasrec_comparison_path = ARTIFACT_DIR / "sasrec_aligned" / "comparison_with_popularity.csv"
if sasrec_comparison_path.exists():
    sasrec_rows = pd.read_csv(sasrec_comparison_path)
    sasrec_rows = sasrec_rows[sasrec_rows["model"] != "popularity"]
    for _, r in sasrec_rows.iterrows():
        rows.append({"model": r["model"], "split": r["split"],
                     "HitRate@10": r["HitRate@10"], "NDCG@10": r["NDCG@10"], "MRR": r["MRR"]})

full_comparison = pd.DataFrame(rows)
display(full_comparison)

full_comparison_path = ARTIFACT_DIR / "full_model_comparison.csv"
full_comparison.to_csv(full_comparison_path, index=False)
print(f"\nSaved: {full_comparison_path}")

,model,split,HitRate@10,NDCG@10,MRR
0,Popularity,val,0.388226,0.256578,0.228811
1,Popularity,test,0.303673,0.198241,0.178813
2,LightFM (purchase-only),val,0.359071,0.247123,0.224746
3,LightFM (purchase-only),test,0.281112,0.190333,0.174696



Saved: artifacts\full_model_comparison.csv
